# H&M GAT seed49 — M1~M4
새 Pod 전용입니다. 기존 GAT·NGCF·Colab은 유지하세요. 이 노트북은 자동으로 GPU를 대여하지 않습니다. 전체 H&M GPU 메모리·시간은 아직 미측정입니다.


## 1. 최초 설치
재접속 시 이미 설치했다면 이 셀은 건너뜁니다. 학습하지 않습니다.


In [ ]:
from pathlib import Path
import subprocess, datetime, json, os
PACKAGE = Path('/workspace/hm_gat_start')
PYTHON = '/workspace/clv/venv/bin/python'
assert (PACKAGE/'launch.py').is_file(), '패키지를 /workspace에 압축 해제하세요.'
subprocess.run(['bash',str(PACKAGE/'setup.sh'),'--job','hm-gat'],check=True)


## 2. 원본 입력 설치·검증
개인 Drive에서 검증한 transactions_train.parquet를 /workspace에 업로드합니다. 같은 볼륨에 원본이 설치돼 있으면 재업로드하지 않습니다.


In [ ]:
from pathlib import Path
import subprocess, datetime, json, os
PACKAGE=Path('/workspace/hm_gat_start')
PYTHON='/workspace/clv/venv/bin/python'
BASE=[PYTHON,'-u',str(PACKAGE/'launch.py')]
OPTIONS=['--job','hm-gat','--seeds','49']
subprocess.run(BASE+['unpack','--bundle',str(PACKAGE/'hm_support.zip')],check=True)
installed=Path('/workspace/clv/data/raw/hm/transactions_train.parquet')
if not installed.exists():
    raw=Path('/workspace/transactions_train.parquet')
    assert raw.is_file(), '원본 업로드가 필요합니다.'
    subprocess.run(BASE+['hm-raw','--file',str(raw)],check=True)
subprocess.run(BASE+['check']+OPTIONS,check=True)


## 3. 실행 범위 확인 후 시작
기본 M1~M4만 순서대로 실행합니다. M5는 추가 지시 후 명시적으로 선택합니다. 다른 곳에서 동일 H&M GAT seed49가 실행 중이면 시작하지 마세요. 원본검사 통과는 전체 그래프 메모리 적합성 보장이 아닙니다. 준비가 실패하면 로그 원인을 확인하며 조건을 자동 변경하지 않습니다.


In [ ]:
MODELS=['M1','M2','M3','M4']
CONFIRMED_NO_DUPLICATE=False
assert CONFIRMED_NO_DUPLICATE, '다른 서버·Colab의 H&M GAT seed49 중복 여부를 확인한 뒤 True로 바꾸세요.'
assert MODELS and len(set(MODELS))==len(MODELS)
assert set(MODELS)<=set(['M1','M2','M3','M4','M5_A','M5_B','M5_C','M5_D'])
logs=Path('/workspace/clv/logs'); logs.mkdir(parents=True,exist_ok=True)
LOG=logs/('hm-gat_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')+'.log')
cmd=BASE+['run']+OPTIONS+['--models',*MODELS,'--confirmed-unstarted']
with LOG.open('x') as output:
    process=subprocess.Popen(cmd,stdout=output,stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,start_new_session=True)
print('실행 요청 PID:',process.pid,'로그:',LOG)
print('실제 시작은 아래 로그의 첫 배치/epoch로 확인하세요. 완료·오류 시에도 Pod 과금은 계속됩니다.')


## 4. 진행 확인
이 셀은 실행할 때마다 최신 로그만 읽습니다. Terminal에서 `tail -f 로그경로`를 실행하면 실시간으로 볼 수 있습니다.


In [ ]:
logs=Path('/workspace/clv/logs')
paths=sorted(logs.glob('hm-gat_*.log'),key=lambda p:p.stat().st_mtime)
assert paths, '시작 로그가 없습니다.'
LOG=paths[-1]
print(LOG)
print(''.join(LOG.read_text(errors='replace').splitlines(True)[-35:]))
print("현재 GPU 사용량(최대 사용량 아님):")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu", "--format=csv"],check=True)


## 5. 결과 위치
`/workspace/clv/exports/hm-gat_results_partial.zip`에 완료된 결과가 저장됩니다. M1~M4만 완료하면 partial 표기가 정상입니다. 8모형 전체 연구계획을 유지합니다. 가중치와 재개 파일은 `/workspace/clv/results/hm-gat`에 따로 보존됩니다. 자동 Pod 정지는 없습니다.
